# D5 序列模型、LSTM 与采样

**竞赛模块：D · 自然语言处理应用开发** · **建议学习约 40 分钟**

今年遇到的 np.random.choice(..., p=preds) 错误就是典型的『模型输出二维矩阵、采样函数要求一维概率』。本节专门练这一点。

> 零基础学习顺序：先读术语和说明 → 运行示例 → 理解代码 → 完成练习 → 概念自测。按 Shift+Enter 运行单元格。

## 一、先认识术语

通过一次真实 LSTM 训练理解为什么模型输出常为二维 (batch,vocab)，以及如何安全交给 NumPy 概率采样。
- **softmax**：把一组分数变成非负且和为 1 的概率
- **采样**：根据概率随机挑选下一个字符/词的索引

RNN/LSTM 按顺序处理序列；模型输出常是二维形状 (batch,vocab)。np.random.choice 的 p 要求一维、非负、和为 1 的概率；应取某一个样本并归一化。

## 二、运行示例

逐格执行并查看输出。这里的代码使用本地/合成数据，遇到缺失输入会有提示。不要直接跳过基础代码。

In [ ]:
import numpy as np
import torch
from torch import nn

# 合成的 4 类符号序列，不下载语料和模型
torch.manual_seed(42)
X = torch.tensor([[0,1,2],[1,2,3],[2,3,0],[3,0,1]])
y = torch.tensor([3,0,1,2])

class TinyLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(4, 8)
        self.lstm = nn.LSTM(8, 12, batch_first=True)
        self.head = nn.Linear(12, 4)

    def forward(self, token_ids):
        embedded = self.embedding(token_ids)
        sequence, _ = self.lstm(embedded)
        return self.head(sequence[:, -1, :])

model = TinyLSTM()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
for epoch in range(40):
    optimizer.zero_grad()
    logits = model(X)
    loss = nn.CrossEntropyLoss()(logits, y)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    logits = model(X[:1])      # (batch=1, vocab=4)
    probabilities = torch.softmax(logits, dim=-1)
probs = probabilities[0].cpu().numpy()  # np.random.choice 只接受一维 p
sample = np.random.choice(len(probs), p=probs)
print("logits:", tuple(logits.shape))
print("概率:", probs.shape, "求和:", probs.sum())
print("采样结果:", sample, "训练损失:", round(loss.item(), 4))

## 三、逐句解释

1. Embedding(4,8) 把 4 种符号编码为 8 维向量；LSTM 的 batch_first=True 接收 (batch,seq,feature)。

2. 每次迭代执行 zero_grad→forward→CrossEntropyLoss→backward→step。

3. X[:1] 得到一个样本，网络输出 logits 形状为 (1,4)。

4. softmax 沿最后一维生成概率；probabilities[0] 去掉 batch 维后转 NumPy。

5. np.random.choice 使用长度为 4 的一维概率 p 采样一个符号索引。


## 四、重点函数与参数

### np.random.choice

- **写法**：`np.random.choice(a, size=None, replace=True, p=None)`
- **参数**：a：候选索引数/数组；p：对应的一维概率数组
- **返回**：返回样本索引或样本数组
- **特别注意**：p 一定要一维、非负、总和近似等于 1

### softmax

- **写法**：`torch.softmax(logits, dim=-1)`
- **参数**：dim=-1：在最后一维按类别归一化
- **返回**：返回与 logits 同 shape 的概率张量
- **特别注意**：NumPy 用 np.exp(x-x.max()) 再除总和也可

## 五、运行结果核验

输出 logits=(1,4)，概率 shape=(4,)，总和约 1，采样索引为 0～3。训练损失随随机初始化及硬件可能略有差异。

> 运行结果与模型随机性有关，请重点核验形状、指标和文件。


## 六、动手实践（先自己写）

**一级：入门**：打印 logits.shape、logits[0].shape、probs.sum()。

**二级：巩固**：将类别扩展到 5 并执行 100 次采样，统计频率。

**三级：挑战**：用 torch.nn.LSTM 输出一个 batch 的 logits，再 softmax 和采样。

**本课小挑战**：打印 logits.shape、probabilities.shape 和 probs.shape；尝试设置 p=probabilities.numpy() 并解释报错，再改回一维。

建议在下面的单元格中独立填写代码，先从一级入门开始，再做二级、三级。

In [ ]:
# TODO 1：完成一级入门练习

# 在这里输入代码


In [ ]:
# TODO 2：完成二级巩固练习

# 在这里输入代码


In [ ]:
# TODO 3：完成三级挑战练习

# 在这里输入代码


## 七、常见错误和排查

1. 直接把 (1,4) 的 probabilities 传入 choice，会报 p must be 1-dimensional。

2. 将 logits 当概率传入 p：logits 可能为负或总和不为 1。

**完成标准**：能够独立解释并运行本课完整示例。


## 八、参考写法（先完成练习再展开）

<details><summary>展开查看原练习参考代码</summary>

```python
probs=np.array([0.2,0.3,0.5])
print(np.random.choice(3,size=10,p=probs))
```

参考写法不一定覆盖全部三级拓展练习；应以实际运行结果为准。

</details>

## 九、概念自测

**问题**：np.random.choice(3,p=...) 的 p 应是？

A. 形状 (1,3) 的 logits

B. 长度 3、和为 1 的一维概率

C. 任意二维矩阵

<details><summary>展开答案与解析</summary>

正确选项：**B**。p 必须是一维概率数组。

</details>

## 十、衔接竞赛实操

学完本模块基础课后，返回学习平台，进入 **6.4.4** 竞赛强化题。

[返回零基础课程网页](../../beginner_course.html#D5)

[打开 6.4.4 竞赛实操题](../../practice.html?question=6.4.4)